# 基础计算定义

## 复制值

$B=A_0$

$\frac{dA}{dt}=0$

$\frac{dB}{dt}=k[A]-k[B]$

In [ ]:
%define COPY(A):(B)
directive parameters [ k = 0.003;]
|A->{k}A+B
|B->{k}
%end define

## 加法

$\frac{dA}{dt}=0$

$\frac{dB}{dt}=0$

$\frac{dY}{dt}=-k[Y]+k([A]+[B])$

In [ ]:
%define ADD0(A,B):(Y)
directive parameters [ k = 0.003;]
|A->{k}A+Y
|B->{k}B+Y
|Y->{k}
%end define

%define ADD(A0,A1,B0,B1):(Y0,Y1)
directive parameters [ k = 0.003;]
%invoke ADD0(A0,B0):(Y0)
%invoke ADD0(A1,B1):(Y1)
|Y0+Y1->{k}
%end define

## 乘法

$Y=A_0*B_0$

$\frac{dA}{dt}=0$

$\frac{dB}{dt}=0$

$\frac{dY}{dt}=k[Y]-k([A]*[B])$

双轨的话

$A*B=(A0-A1)*(B0-B1)=A0*B0-A0*B1-A1*B0+A1*B1$

最终得到$A0*B0+A1*B1 -(A0*B1+A1*B0)$

In [ ]:
%define MUL0(A,B):(Y)
directive parameters [ k = 0.03;]
|A+B->{k}A+B+Y
|Y->{k}
%end define

%define MUL(A0,A1,B0,B1):(Y0,Y1)
directive parameters [ k = 0.03;]
| 0 TMP1
| 0 TMP2
| 0 TMP3
| 0 TMP4
%invoke MUL0(A0,B0):(TMP1)
%invoke MUL0(A1,B1):(TMP2)
%invoke ADD0(TMP1,TMP2):(Y0)
%invoke MUL0(A0,B1):(TMP3)
%invoke MUL0(A1,B0):(TMP4)
%invoke ADD0(TMP3,TMP4):(Y1)
|Y0+Y1->{k}
%end define

# MAC权重乘法相加

In [ ]:
%define MAC0(A,W1,B,W2):(Y)
directive parameters [ k = 0.003;]

| 0 Y1
| 0 Y2
%invoke MUL(A,W1):(Y1)
%invoke MUL(B,W2):(Y2)
%invoke ADD(Y1,Y2):(Y)
%end define

%define MAC(A0,W0,A1,W1,B0,W2,B1,W3):(Y0,Y1)
directive parameters [ k = 0.003;]

| 0 Y3
| 0 Y4
| 0 Y5
| 0 Y6
%invoke MUL(A0,W0,A1,W1):(Y3,Y4)
%invoke MUL(B0,W2,B1,W3):(Y5,Y6)
%invoke ADD(Y3,Y4,Y5,Y6):(Y0,Y1)
%end define

# TANH函数 
来自Design of memory network model based on DNA strand displacement and its application in prediction

调用前需要保证N0,N1其中一个已经湮灭完了

In [ ]:
%define TANH0(N0,N1):(H0,H1)
directive parameters [ k = 0.3;]
| 0 H0
| 0 H1
| N0+N1->{k}
//正分量逻辑 (对应 dH0/dt = k*N0*(1 - H0^2))
|N0->{k}N0+H0
|N0+H0+H0->{k}N0+H0
|N0->{k}
//负分量逻辑 (对应 dH1/dt = k*N1*(1 - H1^2))
|N1->{k}N1+H1
|N1+H1+H1->{k}N1+H1
|N1->{k}
//双轨湮灭 (确保 H0 和 H1 不会同时存在)
|H0+H1->{k}
%end define

# 分子时钟函数

确保反应的顺序执行

参考论文Supervised learning in a multilayer, nonlinear chemical neural network

原理：
$C_{i}$+$C_{i+1}$->{$k_{clock}$}$C_{i+1}$+$C_{i+1}$

外部大时钟

内部小时钟

In [ ]:
%define THREE_CLOCK(c1,c2,c3):(c1,c2,c3)
directive parameters [ k_clock = 0.1 ]
| c1 + c2 ->{k_clock} c2 + c2
| c2 + c3 ->{k_clock} c3 + c3
| c3 + c1 ->{k_clock} c1 + c1
%end define
%define SUB_THREE_CLOCK(c1,c2,c3,CB):(c1,c2,c3,CB)
directive parameters [ k_clock = 0.1 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c1 + CB->{k_clock} c1 + c1+ CB
%end define

## 时钟调控版本

### 时钟加法

In [ ]:
%define ADD0C(A,B,CB):(Y,CB)
directive parameters [ k = 0.003;]
|CB+A->{k}CB+A+Y
|CB+B->{k}CB+B+Y
|CB+Y->{k}CB
%end define
//C1先执行，C2后执行,先加和再湮灭
%define ADDCC(A0,A1,B0,B1,C1,C2):(Y0,Y1,C1,C2)
directive parameters [ k = 0.003;]
%invoke ADD0C(A0,B0,C1):(Y0,C1)
%invoke ADD0C(A1,B1,C1):(Y1,C1)
|C2+Y0+Y1->{k}C2
%end define

或者不需要内部时钟就行

In [ ]:
%define ADDC(A0,A1,B0,B1,CB):(Y0,Y1,CB)
directive parameters [ k = 0.003;]
%invoke ADD0C(A0,B0,CB):(Y0,CB)
%invoke ADD0C(A1,B1,CB):(Y1,CB)
|CB+Y0+Y1->{k}CB
%end define

### 时钟乘法

In [ ]:
%define MUL0C(A,B,CB):(Y,CB)
directive parameters [ k = 0.03;]
|A+B+CB->{k}A+B+Y+CB
|Y+CB->{k}CB
%end define

%define MULCC(A0,A1,B0,B1,C1,C2):(Y0,Y1,C1,C2)
directive parameters [ k = 0.03;]
| 0 TMP1
| 0 TMP2
| 0 TMP3
| 0 TMP4
//一阶段计算没有遇到次序问题
%invoke MUL0C(A0,B0,C1):(TMP1,C1)
%invoke MUL0C(A1,B1,C1):(TMP2,C1)
%invoke MUL0C(A0,B1,C1):(TMP3,C1)
%invoke MUL0C(A1,B0,C1):(TMP4,C1)
%invoke ADD0C(TMP1,TMP2,C1):(Y0,C1)
%invoke ADD0C(TMP3,TMP4,C1):(Y1,C1)
//单独给湮灭上一个周期
|Y0+Y1+C2->{k}C2
%end define
//不需要内部时钟的版本
%define MULC(A0,A1,B0,B1,CB):(Y0,Y1,CB)
directive parameters [ k = 0.03;]
| 0 TMP1
| 0 TMP2
| 0 TMP3
| 0 TMP4
//一阶段计算没有遇到次序问题
%invoke MUL0C(A0,B0,CB):(TMP1,CB)
%invoke MUL0C(A1,B1,CB):(TMP2,CB)
%invoke MUL0C(A0,B1,CB):(TMP3,CB)
%invoke MUL0C(A1,B0,CB):(TMP4,CB)
%invoke ADD0C(TMP1,TMP2,CB):(Y0,CB)
%invoke ADD0C(TMP3,TMP4,CB):(Y1,CB)
|Y0+Y1+CB->{k}CB
%end define

### 时钟MAC

In [ ]:

%define MACC(A0,W0,A1,W1,B0,W2,B1,W3,CB):(Y0,Y1,CB)
directive parameters [ k = 0.003;]

| 0 Y3
| 0 Y4
| 0 Y5
| 0 Y6
%invoke MULC(A0,A1,W0,W1,CB):(Y3,Y4,CB)
%invoke MULC(B0,B1,W2,W3,CB):(Y5,Y6,CB)
%invoke ADDC(Y3,Y4,Y5,Y6,CB):(Y0,Y1,CB)
%end define

### 时钟TANHC
自动湮灭

In [ ]:
%define TANHC(N0,N1,CB):(H0,H1,CB)
directive parameters [ k = 0.2]
| 1 c1
| 1e-6 c2
| 1 c3
%invoke SUB_THREE_CLOCK(c1,c2,c3,CB):(c1,c2,c3,CB)
| c1+N0+N1+CB->{k}c1+CB
//正分量逻辑 (对应 dH0/dt = k*N0*(1 - H0^2))
|c2+N0+CB->{k}c2+N0+H0+CB
|N0+H0+H0+c2+CB->{k}c2+N0+H0+CB
|N0+c2+CB->{k}c2+CB
//负分量逻辑 (对应 dH1/dt = k*N1*(1 - H1^2))
|N1+c2+CB->{k}N1+H1+c2+CB
|N1+H1+H1+c2+CB->{k}N1+H1+c2+CB
|N1+c2+CB->{k}c2+CB
//双轨湮灭 (确保 H0 和 H1 不会同时存在)
|H0+H1+c3+CB->{k}c3+CB
%end define

或许可以考虑一次性的延迟启动也可以，时钟还是循环的，倒也无所谓

# 一个神经元
$tanh(X_1*W_{01}+X_2*W_{11} -W_{00})$

In [ ]:
%define NEURON(X0P,X0N,W0P,W0N,X1P,X1N,W1P,W1N,WP,CB):(YP,YN,CB)
//需要三时钟
| 1 c1
| 1e-6 c2
| 1 c3
| 0 YTMP
| 0 Y0
| 0 Y1
%invoke SUB_THREE_CLOCK(c1,c2,c3,CB):(c1,c2,c3,CB)
//第一步，MAC计算
%invoke MACC(X0P,W0P,X0N,W0N,X1P,W1P,X1N,W1N,c1):(Y0,Y1,c1)
//第二步 WP偏置计算
%invoke ADD0C(Y1,WP,c2):(YTMP,c2)
//第三步tanh计算
%invoke TANHC(Y0,YTMP,c3):(YP,YN,c3)
%end define